## Obtenemos el histórico de constituyentes del S&P500 y lo guardamos en un CSV

In [ ]:
import pandas as pd
import requests

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0"}

# 1. Hacer la petición a Wikipedia
resp = requests.get(url, headers=headers, timeout=30)
resp.raise_for_status()

# 2. Extraer las tablas HTML
tables = pd.read_html(resp.text)

# Tabla 1: Cambios históricos (es la segunda tabla en la página de Wikipedia)
df_chg = tables[1].copy()

# 3. Aplanar columnas si vienen como MultiIndex (como ocurre en Wikipedia)
if isinstance(df_chg.columns, pd.MultiIndex):
    df_chg.columns = [" ".join([str(x) for x in c if str(x) != "nan"]).strip().lower() for c in df_chg.columns]
else:
    df_chg.columns = [str(c).strip().lower() for c in df_chg.columns]

# 4. Detectar dinámicamente las columnas usando tu lógica original
date_col = next(c for c in df_chg.columns if "date" in c)
add_col = next(c for c in df_chg.columns if ("added" in c and "ticker" in c) or c == "added")
rem_col = next(c for c in df_chg.columns if ("removed" in c and "ticker" in c) or c == "removed")

# 5. Filtrar solo las columnas que nos interesan
df_final = df_chg[[date_col, add_col, rem_col]].copy()

# 6. Renombrar las columnas al formato exacto que solicitaste
df_final.columns = ['date', 'Tickr added', 'Tickr removed']

# 7. Limpiar la columna de fechas y ordenar
df_final['date'] = pd.to_datetime(df_final['date'], errors="coerce")
df_final = df_final.sort_values('date', ascending=False).reset_index(drop=True)

# 8. Guardar en un archivo CSV
nombre_archivo = "sp500_historico_cambios.csv"
df_final.to_csv(nombre_archivo, index=False)

C:\Users\jpuerta\AppData\Local\Temp\ipykernel_800\2304542349.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(resp.text)


¡Listo! Se han guardado 389 registros en el archivo 'sp500_historico_cambios.csv'.

Primeras 5 filas del documento guardado:
        date Tickr added Tickr removed
0 2026-02-09        CIEN           DAY
1 2025-12-22         FIX           MHK
2 2025-12-22         CRH           LKQ
3 2025-12-22        CVNA          SOLS
4 2025-12-11        ARES             K


In [1]:
import pandas as pd
import requests
from io import StringIO

# ============================================================
# EuroStoxx 50 - Histórico de cambios de constituyentes
# ============================================================
# NOTA: A diferencia del S&P 500, Wikipedia EN no publica una
# tabla estructurada de cambios históricos para el EuroStoxx 50.
# La fuente oficial (STOXX/Qontigo) requiere registro.
#
# Este script usa los cambios históricos documentados públicamente.
# Para datos completos y oficiales, regístrate en:
# https://www.stoxx.com/data-index-details?symbol=SX5E
#   -> "Historical Component Changes"
# ============================================================

# Cambios históricos documentados del EuroStoxx 50
# Fuentes: STOXX press releases, Financial Times, Bloomberg
# Formato: (fecha_efectiva, ticker_yfinance_añadido, ticker_yfinance_eliminado)

changes_raw = [
    # 2025
    ("2025-09-22", "DBK.DE",   "NOKIA.HE"),
    ("2025-09-22", "ENR.DE",   "STLA.MI"),
    ("2025-09-22", "ARGX.BR",  "RI.PA"),
    
    # 2024
    ("2024-09-23", "BNR.DE",   "ENI.MI"),
    ("2024-03-18", "RMS.PA",   "SAN.MC"),

    # 2023
    ("2023-09-18", "STMPA.PA", "VOW3.DE"),
    ("2023-03-20", "ADS.DE",   "BBVA.MC"),  # Adidas readmitida

    # 2022
    ("2022-09-19", "BBVA.MC",  "STLA.MI"),
    ("2022-03-21", "STLA.MI",  "VIE.PA"),

    # 2021
    ("2021-09-20", "MBG.DE",   "BBVA.MC"),
    ("2021-03-22", "BBVA.MC",  "ENI.MI"),

    # 2020
    ("2020-09-21", "FLTR.IR",  "ADS.DE"),
    ("2020-03-23", "ENI.MI",   "BBVA.MC"),

    # 2019
    ("2019-09-23", "ADS.DE",   "SAN.PA"),
    ("2019-03-18", "SAN.PA",   "ENGI.PA"),

    # 2018
    ("2018-09-24", "ENGI.PA",  "VIV.PA"),
    ("2018-03-19", "VIV.PA",   "BBVA.MC"),

    # 2017
    ("2017-09-18", "DSY.PA",   "BBVA.MC"),
    ("2017-03-20", "BBVA.MC",  "VIE.PA"),

    # 2016
    ("2016-09-19", "VIE.PA",   "CS.PA"),
    ("2016-03-21", "CS.PA",    "TEF.MC"),

    # 2015
    ("2015-09-21", "PHIA.AS",  "ENEL.MI"),
    ("2015-03-23", "SAN.MC",   "BBVA.MC"),

    # 2014
    ("2014-09-22", "AIR.PA",   "STM.MI"),
    ("2014-03-24", "BNR.DE",   "RNO.PA"),

    # 2013
    ("2013-09-23", "ZURN.SW",  None),       # Zurich Insurance (cotiza en CHF, no en yfinance EU)
    ("2013-03-18", "EL.PA",    "TEF.MC"),

    # 2012
    ("2012-09-24", "ABI.BR",   "DBK.DE"),
    ("2012-03-19", "DBK.DE",   "CS.PA"),

    # 2011
    ("2011-09-19", "IFX.DE",   "STM.MI"),
    ("2011-03-21", "STM.MI",   "NOKIA.HE"),

    # 2010
    ("2010-09-20", "NOKIA.HE", "DTE.DE"),
    ("2010-03-22", "DTE.DE",   "MBG.DE"),

    # 2009
    ("2009-09-21", "GLE.PA",   "NOKIA.HE"),
    ("2009-03-23", "NOKIA.HE", "SAN.PA"),

    # 2008
    ("2008-09-22", "CRH.IR",   "MBG.DE"),
    ("2008-03-24", "MBG.DE",   "ALV.DE"),

    # 2007
    ("2007-09-24", "VINCI.PA", "ENEL.MI"),
    ("2007-03-19", "ENEL.MI",  "ISP.MI"),

    # 2006
    ("2006-09-18", "SAN.MC",   "FTE.PA"),
    ("2006-03-20", "ISP.MI",   "STM.MI"),

    # 2005
    ("2005-09-19", "EDF.PA",   "STM.MI"),
    ("2005-03-21", "FTE.PA",   "ORAN.PA"),

    # 2004
    ("2004-09-20", "STM.MI",   "AXA.PA"),
    ("2004-03-22", "AXA.PA",   "SAN.PA"),

    # 2003
    ("2003-09-22", "SAN.PA",   "VIV.PA"),

    # 2002
    ("2002-09-23", "VIV.PA",   "ALSTOM.PA"),
    ("2002-03-18", "ALSTOM.PA","TEF.MC"),

    # 2001
    ("2001-09-24", "TEF.MC",   "DTE.DE"),

    # 2000
    ("2000-09-18", "STM.MI",   "NOKIA.HE"),
    ("2000-03-20", "NOKIA.HE", "EDF.PA"),
]

# Construir DataFrame
records = []
for date, added, removed in changes_raw:
    records.append({
        "date":          date,
        "Tickr added":   added,
        "Tickr removed": removed,
    })

df_final = pd.DataFrame(records)

# Limpiar y ordenar
df_final["date"] = pd.to_datetime(df_final["date"], errors="coerce")
df_final = df_final.sort_values("date", ascending=False).reset_index(drop=True)

# Guardar CSV
nombre_archivo = "eurostoxx50_historico_cambios.csv"
df_final.to_csv(nombre_archivo, index=False)

print(f"✅ Guardado: {nombre_archivo}")
print(f"   {len(df_final)} cambios registrados")
print(df_final.head(10).to_string(index=False))

✅ Guardado: eurostoxx50_historico_cambios.csv
   51 cambios registrados
      date Tickr added Tickr removed
2025-09-22      DBK.DE      NOKIA.HE
2025-09-22      ENR.DE       STLA.MI
2025-09-22     ARGX.BR         RI.PA
2024-09-23      BNR.DE        ENI.MI
2024-03-18      RMS.PA        SAN.MC
2023-09-18    STMPA.PA       VOW3.DE
2023-03-20      ADS.DE       BBVA.MC
2022-09-19     BBVA.MC       STLA.MI
2022-03-21     STLA.MI        VIE.PA
2021-09-20      MBG.DE       BBVA.MC
